# 01 — Phase 0: Day-1 Go/No-Go

This checks whether Mapillary has enough **legible LA parking signs** to build a 500-sign dataset, before any other code is written.

**Steps (from the plan)**
1. ~~Get a Mapillary API token~~ ✓
2. Query one Westwood bounding box ✓, done by `scripts/phase0_probe.py`
3. **Pull 100 images and look at them.** Count the legible parking signs. *(Sections 2–3 below)*
4. **Check the sign-detection layer.** Is it useful for cropping? *(Section 4)*
5. **Hand-transcribe 5 signs with a timer.** *(Section 5)*

**Gate**

| Legible parking signs per 100 images | Action |
|---|---|
| ≥ 10 | **Proceed.** 500 signs is achievable. |
| 3–10 | Proceed, but widen to more neighbourhoods and plan ~3× the query volume. |
| < 3 | **Stop and reconsider.** Try Koreatown / Downtown; if still sparse, revisit the data source. |

**Generating a review sample**
```bash
uv run python scripts/phase0_survey.py                          # pre-screen all target areas, compare yields
uv run python scripts/phase0_probe.py --area koreatown --prescreen   # 100 pre-screened images for review
uv run python scripts/phase0_probe.py --area westwood               # (random, no-filter baseline)
```
Then set `DATA` in the next cell to the folder the probe printed.

**Pre-screen:** an image is shown only if Mapillary detected at least one front-facing sign that is ≥ 60 px tall on the 2048-px scale, portrait-shaped, not cut off by the frame edge, and not overhead; the image is ≥ 1920 px on its long side and not on or beside a freeway. See `src/vlm_parking/prescreen.py`.

> Clear this notebook's outputs before committing. It displays 100 full images.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

from vlm_parking.mapillary import decode_detection_geometry, ring_bbox

DATA = Path("../data/phase0/koreatown_prescreen")   # <- the folder printed by phase0_probe.py
AREA = DATA.name.removesuffix("_prescreen")

def load_json(path, default=None):
    return json.loads(path.read_text()) if path.exists() else default

sample = pd.read_csv(DATA / "sample.csv", dtype={"image_id": str, "sequence": str})
detections = load_json(DATA / "detections.json")
candidates = load_json(DATA / "candidates.json", {})          # detections that passed the pre-screen
population = load_json(DATA / "population.json")              # random-baseline samples only
survey = load_json(Path("../data/phase0/survey") / f"{AREA}.json")
map_features = load_json(DATA / "map_features.json", [])

# Detection classes worth outlining: generic front-facing signs plus any parking-specific classes
SIGN_VALUES = {"object--traffic-sign--front", "object--traffic-sign--information-parking", "object--parking-meter"}
def is_sign(value):
    return value in SIGN_VALUES or value.startswith("regulatory--no-parking") or "parking-restrictions" in value

print(f"{len(sample)} images in {DATA}")

## 1. What the area looks like

For a pre-screened sample, the funnel shows how many of the area's images survive each filter. The review sample is then drawn from the passing images, at most 3 per capture sequence, after dropping repeat views of the same spot.

In [ ]:
if survey:
    print("Pre-screen funnel:", json.dumps(survey["funnel"], indent=1))
if population:
    print("Population by capture year:", population["by_year"])
    print(f"Sequences: {population['n_sequences']}, creators: {population['n_creators']}, panoramas: {population['n_pano']}")
print()
print("Sample by year:", sample.year.value_counts().sort_index().to_dict())
print("Sample by resolution:", sample.groupby(["width", "height"]).size().sort_values(ascending=False).to_dict())

## 2. Review the 100 images

`show_page(k)` displays images `10k … 10k+9`. For each image you get:
- the full frame (downscaled) with Mapillary's sign detections outlined and numbered: **red** = passed the pre-screen, **orange** = other sign detections;
- beneath it, a zoomed crop of each numbered detection with its height in pixels, so you can judge legibility without squinting.

The detector misses signs, so **scan the full frame too**, not just the crops. The Mapillary link opens the original image if you need a closer look.

**What counts:**
- **Parking sign:** any sign governing parking/stopping/standing (no parking, time limits, permit districts, street cleaning, tow-away, loading zones, meters with posted rules).
- **Legible:** you could transcribe *every* panel of it (days, times, limits) with confidence. That's the bar the dataset needs.

In [ ]:
try:
    FONT = ImageFont.load_default(size=28)
except TypeError:
    FONT = ImageFont.load_default()

def sign_boxes(row):
    # (value, bbox, passed_prescreen) in the downloaded thumbnail's pixel coordinates
    passed = {c["id"] for c in candidates.get(row.image_id, [])}
    boxes = []
    for det in detections[row.image_id]:
        if not is_sign(det["value"]):
            continue
        for ring in decode_detection_geometry(det["geometry"], row.img_w, row.img_h):
            boxes.append((det["value"], ring_bbox(ring), det["id"] in passed))
    return sorted(boxes, key=lambda b: not b[2])   # pre-screen passes first

def render(idx, frame_width=1100, crop_height=180, pad=0.35):
    row = sample.loc[idx]
    im = Image.open(DATA / row.path).convert("RGB")
    row = row.copy(); row["img_w"], row["img_h"] = im.size
    boxes = sign_boxes(row)

    crops = []
    for n, (value, (x0, y0, x1, y1), _) in enumerate(boxes):
        w, h = x1 - x0, y1 - y0
        box = (max(0, x0 - pad * w), max(0, y0 - pad * h), min(im.width, x1 + pad * w), min(im.height, y1 + pad * h))
        crop = im.crop(tuple(int(v) for v in box))
        crop = crop.resize((max(1, int(crop.width * crop_height / crop.height)), crop_height), Image.LANCZOS)
        label = f"#{n} {int(h)}px" + (" P" if "parking" in value else "")
        ImageDraw.Draw(crop).text((4, 2), label, fill="yellow", font=FONT, stroke_width=2, stroke_fill="black")
        crops.append(crop)

    frame = im.copy()
    draw = ImageDraw.Draw(frame)
    for n, (_, bb, ok) in enumerate(boxes):
        color = "red" if ok else "orange"
        draw.rectangle(bb, outline=color, width=max(3, im.width // 400))
        draw.text((bb[0], max(0, bb[1] - 34)), f"#{n}", fill=color, font=FONT, stroke_width=2, stroke_fill="white")
    frame = frame.resize((frame_width, int(frame.height * frame_width / frame.width)), Image.LANCZOS)

    print(f"[{idx}] {row.captured_at[:10]}  {row.img_w}x{row.img_h}  pano={row.is_pano}  {row.source_url}")
    display(frame)
    if crops:
        strip = Image.new("RGB", (sum(c.width + 8 for c in crops), crop_height), "white")
        x = 0
        for c in crops:
            strip.paste(c, (x, 0)); x += c.width + 8
        if strip.width > frame_width:
            strip = strip.resize((frame_width, int(crop_height * frame_width / strip.width)), Image.LANCZOS)
        display(strip)
    else:
        print("   (no sign detections)")

def show_page(k, per_page=10):
    for idx in range(k * per_page, min((k + 1) * per_page, len(sample))):
        render(idx)

In [ ]:
PAGE = 0          # 0 … 9; change and re-run to step through all 100 images
show_page(PAGE)

## 3. Record your counts

Fill these in as you go. Use the `[idx]` printed above each image.

In [ ]:
# Images containing at least one parking-related sign, legible or not
parking_sign_idx = []

# Images containing at least one parking sign you could fully transcribe (the number that matters)
legible_idx = []

# Of the legible ones: images where Mapillary's detection box actually covers that sign
detection_covers_idx = []

# Optional free-text notes, e.g. {12: "sign at 40px, tantalisingly close"}
notes = {}

In [ ]:
def gate(rate):
    if rate >= 10: return "PROCEED"
    if rate >= 3:  return "PROCEED, but widen neighbourhoods and plan ~3x query volume"
    return "STOP and reconsider (try Koreatown / Downtown next)"

n = len(sample)
legible_rate = 100 * len(set(legible_idx)) / n
print(f"Any parking sign:      {len(set(parking_sign_idx)):>3} / {n}")
print(f"Legible parking sign:  {len(set(legible_idx)):>3} / {n}  ->  {legible_rate:.0f} per 100")
print(f"Detection covers sign: {len(set(detection_covers_idx) & set(legible_idx)):>3} / {len(set(legible_idx))} legible")
print(f"\nGate: {gate(legible_rate)}")

# Does capture era / camera matter? (recent phone captures vs. 2018 dashcam)
s = sample.assign(legible=sample.idx.isin(legible_idx), era=pd.cut(sample.year, [0, 2019, 2022, 2100], labels=["≤2019", "2020–22", "2023+"]))
display(s.groupby("era", observed=True).legible.agg(n="size", legible="sum"))
display(s.assign(res=s.width.astype(str) + "x" + s.height.astype(str)).groupby("res").legible.agg(n="size", legible="sum").sort_values("n", ascending=False))

### Calibrating the size threshold

The pre-screen used a loose 60 px minimum. Compare the largest passing sign in images you marked legible vs. not. A threshold at roughly the 5th percentile of the legible group keeps ~95% of usable signs while cutting duds; that number goes into the Phase 1 collection pipeline.

In [ ]:
if "max_sign_px" in sample and sample.max_sign_px.gt(0).any():
    grp = sample.assign(legible=sample.idx.isin(legible_idx)).groupby("legible").max_sign_px
    display(grp.describe(percentiles=[0.05, 0.1, 0.25, 0.5]).round(0))
    leg = sample[sample.idx.isin(legible_idx)].max_sign_px
    if len(leg):
        thr = float(leg.quantile(0.05))
        kept_dud = (sample[~sample.idx.isin(legible_idx)].max_sign_px >= thr).mean()
        print(f"Suggested threshold: {thr:.0f} px (keeps 95% of legible, {kept_dud:.0%} of the rest)")
else:
    print("No pre-screen sizes in this sample (random baseline).")

## 4. The sign-detection layer

Two separate Mapillary products:
- **Per-image detections** (outlined above): pixel polygons from Mapillary's segmentation model. Mostly generic classes like `object--traffic-sign--front`, which say *there is a sign here* but not *what kind*.
- **Map features**: detections merged across images into one point per physical sign on the map, classified into Mapillary's traffic-sign taxonomy.

The questions for Phase 1 are whether detection boxes are good enough to **crop** from (you recorded this in `detection_covers_idx`), and whether map features could **find** parking signs directly.

In [ ]:
from collections import Counter

det_values = Counter(d["value"] for dets in detections.values() for d in dets)
print("Per-image detections, sign-related classes (across the 100 images):")
for v, c in det_values.most_common():
    if "sign" in v or "parking" in v or v.startswith(("regulatory", "information")):
        print(f"  {c:>4}  {v}")

heights = []
for idx, row in sample.iterrows():
    r = row.copy(); r["img_w"], r["img_h"] = Image.open(DATA / row.path).size
    heights += [(bb[3] - bb[1]) * 2048 / max(r.img_w, r.img_h) for v, bb, _ in sign_boxes(r) if v == "object--traffic-sign--front"]
heights = pd.Series(heights)
print(f"\nFront-facing sign detections: {len(heights)}; height in px (on the 2048-px thumbnail):")
print(heights.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(0).to_string())

In [ ]:
fv = Counter(f["object_value"] for f in map_features)
types = Counter(f.get("object_type") for f in map_features)
print(f"Map features in bbox: {len(map_features):,} ({dict(types)}), {len(fv)} distinct values")
print("\nParking-related map features:")
for v, c in fv.most_common():
    if any(h in v for h in ("parking", "stopping", "standing", "tow", "loading", "permit")):
        print(f"  {c:>4}  {v}")
print("\nTop 15 traffic-sign values overall:")
for v, c in Counter(f["object_value"] for f in map_features if f.get("object_type") == "trafficsign").most_common(15):
    print(f"  {c:>4}  {v}")

## 5. Hand-transcribe 5 signs, timed

Pick 5 legible signs from your review, ideally a mix of 1-panel and multi-panel. For each one, call `start(idx)`, transcribe the sign into the `transcriptions` cell in the schema's spirit (one line per panel: rule, days, times, limit, district), then call `stop(idx)`.

The time per sign sets the labeling budget: at 500 signs, 60 s/sign is ~8 hours; 120 s/sign is ~17 hours.

In [ ]:
import time
_timers, timings = {}, {}

def start(idx):
    _timers[idx] = time.time()
    render(idx)

def stop(idx):
    timings[idx] = round(time.time() - _timers.pop(idx))
    print(f"[{idx}] {timings[idx]} s")

In [ ]:
# start(12)   # replace 12 with the idx of a legible sign

In [ ]:
# stop(12)

In [ ]:
# One entry per sign; one line per panel, top to bottom
transcriptions = {
    # 12: '''
    # no_parking | TUE | 08:00-10:00 | street cleaning
    # time_limited 120 | MON-SAT | 08:00-18:00
    # permit_only district 7 | exempt from time limit
    # ''',
}

## 6. Save the day-1 numbers

These go in the README; they justify every later design choice.

In [ ]:
assert legible_idx or parking_sign_idx, "Fill in the lists in Section 3 before saving results"
t = pd.Series(timings, dtype=float)
results = {
    "data": str(DATA),
    "prescreen_funnel": survey["funnel"] if survey else None,
    "sample_n": n,
    "any_parking_sign": len(set(parking_sign_idx)),
    "legible_parking_sign": len(set(legible_idx)),
    "legible_per_100": round(legible_rate, 1),
    "detection_covers_legible": len(set(detection_covers_idx) & set(legible_idx)),
    "gate": gate(legible_rate),
    "transcription_seconds": timings,
    "transcription_median_s": None if t.empty else float(t.median()),
    "notes": {str(k): v for k, v in notes.items()},
}
(DATA / "results.json").write_text(json.dumps(results, indent=1))
print(json.dumps(results, indent=1))